In [ ]:
import pandas as pd
import numpy as np
import openai
import ast
import os

In [10]:
file_path = r"C:\Users\msn-f\Desktop\Comments.csv"
# file_path = r"C:\Users\msn-f\Desktop\comments_filtered.csv"
df = pd.read_csv(file_path)
display(df)

,a_Ministry_Family,b_Agency,c_Length_Of_Service,d_Main Question,e_Main Question Description,f_Sub Question,g_Sub Question Description,h_Other Applications / Initiatives,i_Question Type,j_Rated,k_Verbatims
0,MDDI,GOVTECH,1-5 Yr,Q1,Q1. Please rate your overall level of satisfac...,Q1,NaN,NaN,For Improvement,4 - Slightly Satisfied,NaN
1,MDDI,GOVTECH,1-5 Yr,Q1,Q1. Please rate your overall level of satisfac...,Q1,NaN,NaN,For Improvement,5 - Satisfied,NaN
2,MDDI,GOVTECH,1-5 Yr,Q1,Q1. Please rate your overall level of satisfac...,Q1,NaN,NaN,For Improvement,6 - Very Satisfied,NaN
3,MDDI,GOVTECH,6-9 Yr,Q1,Q1. Please rate your overall level of satisfac...,Q1,NaN,NaN,For Improvement,5 - Satisfied,NaN
4,MDDI,GOVTECH,>= 10 Yr,Q1,Q1. Please rate your overall level of satisfac...,Q1,NaN,NaN,For Improvement,5 - Satisfied,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7324,MDDI,GOVTECH,1-5 Yr,Q10,Q10. How satisfied are you with the engagement...,Q10(b),Product Training & Brownbag (Virtual & F2F),NaN,Most Effective,5 - Satisfied,"Teams channel posts, Brownbag"
7325,MDDI,GOVTECH,6-9 Yr,Q10,Q10. How satisfied are you with the engagement...,Q10(b),Product Training & Brownbag (Virtual & F2F),NaN,Most Effective,5 - Satisfied,Technical FAQ channel on SG-Teams is quite hel...
7326,MDDI,GOVTECH,1-5 Yr,Q10,Q10. How satisfied are you with the engagement...,Q10(b),Product Training & Brownbag (Virtual & F2F),NaN,Most Effective,4 - Slightly Satisfied,Too many eDM mailers and often the email messa...
7327,MDDI,GOVTECH,>= 10 Yr,Q10,Q10. How satisfied are you with the engagement...,Q10(b),Product Training & Brownbag (Virtual & F2F),NaN,Most Effective,6 - Very Satisfied,"Virtual product training, eDMs."


In [ ]:
# Pass the API Key to the OpenAI Client
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
# Some other code here are omitted for brevity

# --- 3. AI Call (OpenAI GPT example) ---
def get_ai_response(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.choices[0].message.content

In [4]:
# Classification function
def classify_rating(value):
    if str(value).strip().startswith(('1', '2', '3', '4')):
        return 'unsatisfied'
    elif str(value).strip().startswith(('5', '6')):
        return 'satisfied'
    else:
        return 'unknown'

In [46]:
def build_prompt_extract_groupings(main_q, sub_q, verbatims):
    prompt = f"""
You are an analyst reviewing customer feedback for an internal tools survey.

Here is the context:
- Main Question: {main_q}
- Sub Question: {sub_q}

Below is a verbatim comment from users:
{verbatims}

Here are the categories you should classify the comments into:
categories:
["Outlook", "EXO", "COMET", "GSIB", "GOMAX", "OneSpace (KM) / Intranet", "AMS", "RBS", "BookingSG", "Onedrive", "SPO", "SGTeams", "SG-DCS", "IT Support"]

Detail of each category is as follows:
1. Outlook: Email and calendar services.
2. EXO: Exchange Online, part of Microsoft 365 product.
3. COMET: Laptop that is after migration. Device that works using cloud solutions services and most data is stored in the cloud.
4. GSIB: Government Secured Intranet Device. Laptop used before migration to COMET.
5. GOMAX: Mobile device provision to certain users.
6. OneSpace (KM): Knowledge management platform. Internal company network and resources are uploaded here.
8. AMS: Asset Management System. Used for managing IT assets.
9. RBS: Room Booking System. Used for booking meeting rooms.
10. BookingSG: Booking system used in another building called Hive. Used for applying leave and meeting rooms for users in Hive.
11. Onedrive: Cloud storage service.
12. SPO: SharePoint Online, part of Microsoft 365.
13. SGTeams: Microsoft Teams for Singapore Government.
14. SG-DCS: Singapore Data Center Services. Similar to Onedrive but for higher classification data.
14. IT Support: Technical support services. 


Please classify the comments into the categories above. If a comment does not fit any category, please return "Other".
If the verbatim talks about multiple categories, please seperate the verbatims and the category return them in the format below.

Return your response in this format:
{{"verbatim1": "category1",
"verbatim2": "category2", ...}}
"""
    return prompt.strip()

In [39]:
df2 = df[['e_Main Question Description','g_Sub Question Description','h_Other Applications / Initiatives','k_Verbatims']]

df3 = df2[df2['k_Verbatims'].notna() & (df2['k_Verbatims'].astype(str).str.strip() != '')]

df3["full_verbatim"] = (
            "apps mention: " + df3["h_Other Applications / Initiatives"].fillna("None") +
            " - verbatim provided by user: " + df3["k_Verbatims"].fillna("None")
        ).copy()

C:\Users\msn-f\AppData\Local\Temp\ipykernel_22520\3016133150.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["full_verbatim"] = (


In [40]:
test_df = df3.head().copy()

In [41]:
test_df

,e_Main Question Description,g_Sub Question Description,h_Other Applications / Initiatives,k_Verbatims,full_verbatim
6,Q1. Please rate your overall level of satisfac...,NaN,NaN,"Fragmented tools, troublesome onboarding with...",apps mention: None - verbatim provided by user...
7,Q1. Please rate your overall level of satisfac...,NaN,NaN,Once in a while the Wifi in the office is bad...,apps mention: None - verbatim provided by user...
8,Q1. Please rate your overall level of satisfac...,NaN,NaN,1 device instead of bring 2 devices to work ev...,apps mention: None - verbatim provided by user...
9,Q1. Please rate your overall level of satisfac...,NaN,NaN,"1. For COMET laptops, connectivity breaks when...",apps mention: None - verbatim provided by user...
10,Q1. Please rate your overall level of satisfac...,NaN,NaN,1. Leeway to whitelist online sites and social...,apps mention: None - verbatim provided by user...


In [55]:
# Store results here
results = []

for idx, row in df3.iterrows():
    main_question = row['e_Main Question Description']
    sub_question = row['g_Sub Question Description']
    verbatim = row['full_verbatim']
    
    print(f"Processing row {idx}: {main_question}, {sub_question}, {verbatim}")
    try:
        # Call your AI function
        prompt = build_prompt_extract_groupings(main_question, sub_question, verbatim)
        ai_output = get_ai_response(prompt)
        print(f"AI output for row {idx}: {ai_output}")
        # Get the AI response
        # Ensure the result is in dictionary format
        if isinstance(ai_output, str):
            output = ast.literal_eval(ai_output)  # safely convert string to dict
        
        for extracted_verbatim, category in output.items():
            new_row = row.to_dict()
            new_row['verbatim_extracted'] = extracted_verbatim
            new_row['category'] = category
            results.append(new_row)
    except Exception as e:
        print(f"Error processing row {idx}: {e}")

# Create a new dataframe with the results
df_result = pd.DataFrame(results)

# Optional: Reset index
df_result.reset_index(drop=True, inplace=True)

Processing row 6: Q1. Please rate your overall level of satisfaction with the ICT Tools & Services available to support your work., nan, apps mention: None - verbatim provided by user:  Fragmented tools, troublesome onboarding with multiple parties.
AI output for row 6: {"Fragmented tools, troublesome onboarding with multiple parties.": "Other"}
Processing row 7: Q1. Please rate your overall level of satisfaction with the ICT Tools & Services available to support your work., nan, apps mention: None - verbatim provided by user:  Once in a while the Wifi in the office is bad. Only once in a while. But the IT engineer comes down dutifully after every complaint and tries to measure it but the signal is OK by then, so no one really knows why or how to fix it. 
AI output for row 7: {"Once in a while the Wifi in the office is bad. Only once in a while. But the IT engineer comes down dutifully after every complaint and tries to measure it but the signal is OK by then, so no one really knows wh

In [56]:
df_result

,e_Main Question Description,g_Sub Question Description,h_Other Applications / Initiatives,k_Verbatims,full_verbatim,verbatim_extracted,category
0,Q1. Please rate your overall level of satisfac...,NaN,NaN,"Fragmented tools, troublesome onboarding with...",apps mention: None - verbatim provided by user...,"Fragmented tools, troublesome onboarding with ...",Other
1,Q1. Please rate your overall level of satisfac...,NaN,NaN,Once in a while the Wifi in the office is bad...,apps mention: None - verbatim provided by user...,Once in a while the Wifi in the office is bad....,IT Support
2,Q1. Please rate your overall level of satisfac...,NaN,NaN,1 device instead of bring 2 devices to work ev...,apps mention: None - verbatim provided by user...,1 device instead of bring 2 devices to work ev...,Other
3,Q1. Please rate your overall level of satisfac...,NaN,NaN,"1. For COMET laptops, connectivity breaks when...",apps mention: None - verbatim provided by user...,"For COMET laptops, connectivity breaks when id...",COMET
4,Q1. Please rate your overall level of satisfac...,NaN,NaN,"1. For COMET laptops, connectivity breaks when...",apps mention: None - verbatim provided by user...,"For COMET laptops, it would be helpful to enab...",COMET
...,...,...,...,...,...,...,...
8539,Q10. How satisfied are you with the engagement...,Product Training & Brownbag (Virtual & F2F),NaN,Technical FAQ channel on SG-Teams is quite hel...,apps mention: None - verbatim provided by user...,Technical FAQ channel on SG-Teams is quite hel...,SGTeams
8540,Q10. How satisfied are you with the engagement...,Product Training & Brownbag (Virtual & F2F),NaN,Too many eDM mailers and often the email messa...,apps mention: None - verbatim provided by user...,Too many eDM mailers and often the email messa...,Outlook
8541,Q10. How satisfied are you with the engagement...,Product Training & Brownbag (Virtual & F2F),NaN,"Virtual product training, eDMs.",apps mention: None - verbatim provided by user...,Virtual product training,Other
8542,Q10. How satisfied are you with the engagement...,Product Training & Brownbag (Virtual & F2F),NaN,"Virtual product training, eDMs.",apps mention: None - verbatim provided by user...,eDMs,Other


In [57]:
# Export to Excel
output_path = r"C:\Users\msn-f\Desktop\comments_category_selection.xlsx"
df_result.to_excel(output_path, index=False)

In [5]:
def build_prompt(main_q, sub_q, classification, app_str, verbatims):
    verbatim_list = "\n".join(f"- {v}" for v in verbatims)
    prompt = f"""
You are an analyst reviewing customer feedback for an internal tools survey.

Here is the context:
- Main Question: {main_q}
- Sub Question: {sub_q}
- Classification: {classification}
- Question Type: {app_str}

Below is a list of verbatim comments from users:
{verbatim_list}

Please do the following:
1. Identify and return 2–5 key **themes or categories** expressed in the comments (as a Python list).
2. Provide a **summary** of the feedback in 2–5 clear bullet points, capturing user sentiment and most common key pain points or suggestions.

Return your response in this format:

categories:
["category 1", "category 2", ...]

summary:
• bullet point 1  
• bullet point 2  
• bullet point 3
"""
    return prompt.strip()

In [10]:
def build_prompt(main_q, sub_q, classification, app_str, verbatims):
    verbatim_list = "\n".join(f"- {v}" for v in verbatims)
    prompt = f"""
You are an analyst reviewing customer feedback for an internal tools survey.

Here is the context:
- Main Question: {main_q}
- Sub Question: {sub_q}
- Classification: {classification}
- Question Type: {app_str}

Below is a list of verbatim comments from users:
{verbatim_list}

Please do the following:
1. Identify and return 2–5 key **themes or categories** expressed in the comments (as a Python list).
2. Separate the feedback into two parts:
   a. **Pain Points** – What users think is not working well or causing frustration.
   b. **Suggestions** – Ideas or requests from users on how things could be improved.

Return your response in this format:

categories:
["category 1", "category 2", "category 3", ...]

pain_points:
• bullet point 1  
• bullet point 2  
• bullet point 3

suggestions:
• bullet point 1  
• bullet point 2  
• bullet point 3
"""
    return prompt.strip()

In [4]:
def extract_build_prompt(verbatims):
    verbatim_list = "\n".join(f"- {v}" for v in verbatims)
    prompt = f"""
You are given a list of open-text survey comments. Your task is to extract and return only the comments that mention support services or the EU Settlement Scheme (EUSS).
This includes any references to:

Help, support, or assistance (e.g., "I was helped", "support from staff", "got assistance", etc.)

EUSS (End User Support Service)

Instructions:

Be case-insensitive

Include comments even if the mention is implicit (e.g., "they helped me apply" implies support with EUSS)

Return only the relevant comments as a list

Here is the input list of comments:
{verbatim_list}
"""
    return prompt.strip()

In [11]:
def split_list_into_parts(lst, num_parts):
    k, m = divmod(len(lst), num_parts)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(num_parts)]

In [13]:
# Extract non-null verbatims from the dataframe
verbatims = df["k_Verbatims"].dropna().tolist()
verbatims[:10]  # Display first 10 verbatims as a sample

lis = []
parts = split_list_into_parts(verbatims, 40)
for i, part in enumerate(parts):
    prompt = extract_build_prompt(part)

    ai_output = get_ai_response(prompt)

    lis.append(ai_output)

In [16]:
# Flatten the 'lis' object (which is a list of lists or strings) into a single list
flattened_lis = []
for item in lis:
    if isinstance(item, list):
        flattened_lis.extend(item)
    else:
        flattened_lis.append(item)

In [21]:
def extract_build_prompt(verbatims):
    verbatim_list = "\n".join(f"- {v}" for v in verbatims)
    prompt = f"""
You are an analyst reviewing customer feedback for an internal tools survey.

Below is a list of verbatim comments from users:
{verbatim_list}


Please do the following:
1. Summarize the comments in 2–5 clear bullet points, capturing user sentiment on what is done well and what could be improved.
"""
    return prompt.strip()

In [24]:
lis2 = []
for item in lis:
    prompt = extract_build_prompt(item)

    ai_output = get_ai_response(prompt)

    lis2.append(ai_output)

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens. However, your messages resulted in 55599 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [6]:
def clean_apps_prompt(apps):
    apps = str(apps).strip()
    prompt = f"""
You are an analyst reviewing customer feedback for an internal tools survey.
You are provided with a list of applications mentioned in the comments.
Please clean the list by removing any duplicates, irrelevant entries, or empty values.

Return your response in this format:
["Application 1", "Application 2", ...]
    """
    
    output = get_ai_response(prompt)

    return output

In [21]:
# --- 4. Grouping + Pipeline ---
def analyze_feedback(df):
    df["classification"] = df["j_Rated"].apply(classify_rating)

    group_keys = ["i_Question Type", "e_Main Question Description" ,"f_Sub Question", "classification"]

    results = []

    grouped = df.groupby(group_keys)

    for keys, group in grouped:
        qns_type, main_q, sub_q, cls = keys
        sub_q = group.get("g_Sub Question Description", "N/A").unique()  # if you don't have this, skip
        group = group.dropna(subset=['k_Verbatims'])

        group["full_verbatim"] = (
            "apps mention: " + group["h_Other Applications / Initiatives"].fillna("None") +
            " - verbatim provided by user: " + group["k_Verbatims"].fillna("None")
        )

        verbatims = group["full_verbatim"].dropna().tolist()
        if not verbatims:
            continue
        print(main_q, sub_q, cls, qns_type, verbatims)
        # build_prompt(main_q, sub_q, classification, app_str, verbatims)
        prompt = build_prompt(main_q, sub_q, cls, qns_type, verbatims)
        try:
            ai_output = get_ai_response(prompt)
            print(f"AI Output for {keys}: {ai_output}")
            # Basic parse: assume categories: [...] and summary: bullets
            categories = []
            summary = []
            print(summary)

            # Very basic parsing logic:
            if "categories:" in ai_output.lower():
                parts = ai_output.split("categories:")[1].split("pain_points:")
                cat_block = parts[0].strip()
                sum_block = parts[1].strip()

                categories = eval(cat_block) if cat_block.startswith("[") else [cat_block]
                summary = sum_block.strip()
                # print(ai_output)
            
            results.append({
                "Main Question": main_q,
                "Sub Question": sub_q,
                "classification": cls,
                "question type": qns_type,
                "categories": categories,
                "summary": summary
            })

        except Exception as e:
            print(f"Failed to process group {keys}: {e}")

    return pd.DataFrame(results)

In [ ]:
# Run analysis
results_df = analyze_feedback(df)

In [23]:
len(results_df)

20

In [24]:
# Export to Excel
output_path = r"C:\Users\msn-f\Desktop\Comments_drilldown_summary_top_5_points.xlsx"
results_df.to_excel(output_path, index=False)